In [ ]:
import pygame
import numpy as np
import sys

pygame.init()

WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("PolySnake: Classic Growth + No Teleport")
clock = pygame.time.Clock()

BACKGROUND = (10, 10, 30)
SNAKE_COLOR = (0, 255, 150)
POINT_COLOR = (255, 60, 60)
PATH_COLOR = (100, 200, 255)
TEXT_COLOR = (220, 220, 220)

# Snake body: list of segments (tail to head)
snake_body = [(WIDTH // 2, HEIGHT // 2)] * 10  # start with 10 segments
points = []
full_path = []  # complete path: [linear_to_start] + [polynomial]
moving = False
degree = 3
EAT_RADIUS = 18
SEGMENTS_PER_FOOD = 10  # classic growth

def generate_full_path(snake_head, pts):
    """Generate a smooth path: from snake head → through all points via polynomial."""
    if not pts:
        return []
    
    # Step 1: Fit polynomial through the user points only
    xs = np.array([p[0] for p in pts])
    ys = np.array([p[1] for p in pts])
    
    try:
        deg = min(degree, len(pts) - 1) if len(pts) > 1 else 0
        if len(pts) == 1:
            poly_path = [pts[0]]
        else:
            coeffs = np.polyfit(xs, ys, deg)
            poly = np.poly1d(coeffs)
            x_min, x_max = xs.min(), xs.max()
            x_smooth = np.linspace(x_min, x_max, 300)
            y_smooth = poly(x_smooth)
            poly_path = list(zip(x_smooth, y_smooth))
    except:
        # Fallback: just go to points in order
        poly_path = pts[:]
    
    if not poly_path:
        return []
    
    # Step 2: Create linear segment from snake_head to first poly point
    first_point = poly_path[0]
    connect_steps = 60  # enough for smooth approach at normal speed
    connect_path = []
    for t in range(connect_steps):
        alpha = t / connect_steps
        x = snake_head[0] + alpha * (first_point[0] - snake_head[0])
        y = snake_head[1] + alpha * (first_point[1] - snake_head[1])
        connect_path.append((x, y))
    
    # Step 3: Combine
    return connect_path + poly_path

def distance(p1, p2):
    return ((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2) ** 0.5

# Main loop
running = True
path_index = 0

while running:
    mouse_pos = pygame.mouse.get_pos()
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            points.append(mouse_pos)
        
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 3:
            if points:
                head = snake_body[-1]
                full_path = generate_full_path(head, points)
                path_index = 0
                moving = True
        
        if event.type == pygame.KEYDOWN and event.key == pygame.K_r:
            snake_body = [(WIDTH // 2, HEIGHT // 2)] * 10
            points = []
            full_path = []
            moving = False
            path_index = 0

    # --- Movement ---
    if moving and full_path:
        if path_index < len(full_path):
            new_head = full_path[path_index]
            path_index += 1  # consistent speed: 1 step per frame
            
            # Wrap around screen
            new_head = (new_head[0] % WIDTH, new_head[1] % HEIGHT)
            
            # Check for eaten points
            eaten = False
            remaining_points = []
            for p in points:
                if distance(new_head, p) <= EAT_RADIUS:
                    eaten = True
                else:
                    remaining_points.append(p)
            points = remaining_points

            # Update snake body
            snake_body.append(new_head)
            if not eaten:
                # Normal move: remove tail
                if len(snake_body) > 1:
                    snake_body.pop(0)
            else:
                # Grow: keep tail + add extra segments
                for _ in range(SEGMENTS_PER_FOOD - 1):
                    snake_body.insert(0, snake_body[0])  # extend tail
        else:
            moving = False  # finished

    # --- DRAWING ---
    screen.fill(BACKGROUND)
    
    # Draw full path while moving
    if moving and len(full_path) > 1:
        wrapped_path = [(x % WIDTH, y % HEIGHT) for x, y in full_path]
        pygame.draw.lines(screen, PATH_COLOR, False, wrapped_path, 2)
    
    # Draw points
    for p in points:
        pygame.draw.circle(screen, POINT_COLOR, p, 12)
        pygame.draw.circle(screen, (255, 200, 200), p, 12, 2)
    
    # Draw snake
    if len(snake_body) > 1:
        wrapped_body = [(x % WIDTH, y % HEIGHT) for x, y in snake_body]
        pygame.draw.lines(screen, SNAKE_COLOR, False, wrapped_body, 16)
    if snake_body:
        head = (snake_body[-1][0] % WIDTH, snake_body[-1][1] % HEIGHT)
        pygame.draw.circle(screen, (0, 255, 255), head, 10)

    # UI
    font = pygame.font.SysFont(None, 24)
    instructions = [
        "LEFT CLICK: Add point",
        "RIGHT CLICK: Fit & move snake along polynomial",
        "R: Reset",
        f"Points: {len(points)} | Length: {len(snake_body)}"
    ]
    for i, line in enumerate(instructions):
        text = font.render(line, True, TEXT_COLOR)
        screen.blit(text, (10, 10 + i * 25))

    pygame.display.flip()
    clock.tick(60)

pygame.quit()
sys.exit()